# Download Kaggle Dataset

In [1]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("asad1m9a9h6mood/news-articles")

#print("Path to dataset files:", path)

In [2]:
import os

# Dataset location
path = "/Users/anushka/.cache/kagglehub/datasets/asad1m9a9h6mood/news-articles/versions/1"

# Display all files in the dataset folder
for root, dirs, files in os.walk(path):
    for file in files:
        print(os.path.join(root, file))

/Users/anushka/.cache/kagglehub/datasets/asad1m9a9h6mood/news-articles/versions/1/Articles.csv


# Exploring the Dataset

In [3]:
import pandas as pd

# Dataset location
file_path = "/Users/anushka/.cache/kagglehub/datasets/asad1m9a9h6mood/news-articles/versions/1/Articles.csv"

# Load dataset
df = pd.read_csv(file_path, encoding="latin1")
df.head()

,Article,Date,Heading,NewsType
0,KARACHI: The Sindh government has decided to b...,1/1/2015,sindh govt decides to cut public transport far...,business
1,HONG KONG: Asian markets started 2015 on an up...,1/2/2015,asia stocks up in new year trad,business
2,HONG KONG: Hong Kong shares opened 0.66 perce...,1/5/2015,hong kong stocks open 0.66 percent lower,business
3,HONG KONG: Asian markets tumbled Tuesday follo...,1/6/2015,asian stocks sink euro near nine year,business
4,NEW YORK: US oil prices Monday slipped below $...,1/6/2015,us oil prices slip below 50 a barr,business


In [4]:
print("Total number of Articles:", len(df))
df['Date'] = pd.to_datetime(df['Date'])
print("Earliest date:", df['Date'].min())
print("Latest date:", df['Date'].max())
print("\nType of Articles:", df['NewsType'].value_counts())
print("\nMissing values:", df.isnull().sum())
print("\nCheck for duplicates:", df['Article'].duplicated().sum())

Total number of Articles: 2692
Earliest date: 2015-01-01 00:00:00
Latest date: 2017-03-27 00:00:00

Type of Articles: NewsType
sports      1408
business    1284
Name: count, dtype: int64

Missing values: Article     0
Date        0
Heading     0
NewsType    0
dtype: int64

Check for duplicates: 108


# Data Preprocessing

In [5]:
# Using a copy of the dataset to preserve the original.
df_clean = df.copy()

# Remove duplicates. 
df_clean = df_clean.drop_duplicates()

# Check for whitespace
whitespace_check = df[ df['Article'] != df['Article'].str.strip()]
print("Articles with leading or trailing whitespace:", len(whitespace_check))

# Strip whitespace
df_clean['Article'] = df_clean['Article'].str.strip()
df_clean['Heading'] = df_clean['Heading'].str.strip()

# Check for empty Articles. Did this before but just to make sure it does not contain empty strings.
empty_articles = df[ df['Article'].isna() | (df['Article'].str.strip() == '')]
print("Number of empty articles:", len(empty_articles))

# Sort by date
df_clean = df_clean.sort_values('Date', ascending = False)
print("Articles after cleaning:", len(df_clean))

Articles with leading or trailing whitespace: 890
Number of empty articles: 0
Articles after cleaning: 2585


In [6]:
# Select random Articles
# 5 samples from each Newstype. 
# Same Articles choosen every run due to random_state
sample = (
    df.drop_duplicates(subset='Article')
      .groupby('NewsType', group_keys=False)
      .sample(n=5, random_state=42) 
      .reset_index(drop=True)
)

# Assign unique IDs
sample['Article_id'] = range(0, len(sample))

print("Selected articles:", len(sample))
print("Amount for each type of Article:", sample['NewsType'].value_counts())

sample

Selected articles: 10
Amount for each type of Article: NewsType
business    5
sports      5
Name: count, dtype: int64


,Article,Date,Heading,NewsType,Article_id
0,"London: Oil prices nudged higher on Friday, fu...",2015-04-10,oil prices edge higher,business,0
1,strong>SINGAPORE: Oil prices stabilized on Wed...,2016-05-04,Oil stable after two day decline on stall in g...,business,1
2,KARACHI: The announcement of Monetary Policy f...,2015-03-16,further cut in sbp policy rate expected,business,2
3,London: Crude prices fell Thursday awaiting th...,2015-10-15,oil prices fall before us inventory d,business,3
4,strong>SAN FRANCISCO/NEW YORK: U.S. Federal Re...,2016-09-20,Fed again poised to cut longer run interest ra...,business,4
5,"strong>PALLEKELE, Sri Lanka: Australia´s Josh ...",2016-07-26,Australia skittle out Sri Lanka just 117,sports,5
6,strong>SYDNEY: Australian spin great Shane War...,2016-03-31,Warne blasts Australia selectors for latest T2...,sports,6
7,strong>LONDON: Azhar Ali´s fine form with the ...,2016-07-09,In form Ali boosts Paki,sports,7
8,BANGALORE: Bangladesh captain Mashrafe Mortaza...,2016-03-23,Bangladesh win toss field against Indi,sports,8
9,ISLAMABAD: Pakistan Cricket Board (PCB) is con...,2016-04-27,PCB conducting dope tests on directives of ICC,sports,9


In [7]:
# Since some Articles contain HTML tags, I shall be removing them.
html_check = sample[
    sample['Article'].str.contains(r'<[^>]+>|strong>', regex=True, na=False)
]

print("Articles containing possible HTML tags:", len(html_check))


for _, row in html_check.iterrows():
    print("\nArticle ID:", row['Article_id'])
    print(row['Article'][:100])

Articles containing possible HTML tags: 5

Article ID: 1
strong>SINGAPORE: Oil prices stabilized on Wednesday after falling for two straight days on concerns

Article ID: 4
strong>SAN FRANCISCO/NEW YORK: U.S. Federal Reserve policymakers are set this week to again cut thei

Article ID: 5
strong>PALLEKELE, Sri Lanka: Australia´s Josh Hazlewood and Nathan Lyon led an inspired bowling atta

Article ID: 6
strong>SYDNEY: Australian spin great Shane Warne has blamed his country´s latest World Twenty20 fail

Article ID: 7
strong>LONDON: Azhar Ali´s fine form with the bat gave Pakistan the perfect boost ahead of the first


In [8]:
# Removes HTML tags like /strong or strong
sample['Article'] = sample['Article'].str.replace(r'</?strong>','', regex=True, case=False)

sample['Article'] = sample['Article'].str.replace( r'strong>', '', regex=True, case=False)

In [9]:
# Check cleaned up Articles
for Article_id in [1, 4, 5, 6, 7]:
    article = sample.loc[sample['Article_id'] == Article_id, 'Article'].iloc[0]

    print("\nArticle ID:", Article_id)
    print(repr(article[:100]))


Article ID: 1
'SINGAPORE: Oil prices stabilized on Wednesday after falling for two straight days on concerns that s'

Article ID: 4
'SAN FRANCISCO/NEW YORK: U.S. Federal Reserve policymakers are set this week to again cut their forec'

Article ID: 5
'PALLEKELE, Sri Lanka: Australia´s Josh Hazlewood and Nathan Lyon led an inspired bowling attack to h'

Article ID: 6
'SYDNEY: Australian spin great Shane Warne has blamed his country´s latest World Twenty20 failure on '

Article ID: 7
'LONDON: Azhar Ali´s fine form with the bat gave Pakistan the perfect boost ahead of the first Test w'
